# 09 - Deployment: FastAPI + Streamlit
The trained artifacts are served through two interfaces:
- **REST API** - `uvicorn src.api.fastapi_app:app --reload`
- **Dashboard** - `streamlit run frontend/dashboard.py`

> This notebook smoke-tests every endpoint in-process via FastAPI's TestClient.

## 1. Create the app and hit /health + /symbols

In [ ]:
from fastapi.testclient import TestClient

from src.api.fastapi_app import app

client = TestClient(app)
print(client.get('/api/v1/health').json())
print(client.get('/api/v1/symbols').json()['ready'])

## 2. Prediction / decision / risk endpoints

In [ ]:
for ep in ('predict','decision','risk'):
    r = client.get(f'/api/v1/{ep}/NVDA')
    body = r.json()
    print(ep, r.status_code, str(body)[:110])

## 3. Explanation endpoint (SHAP + attention)

In [ ]:
r = client.get('/api/v1/explain/NVDA')
print(r.status_code, str(r.json())[:220])

## 4. Portfolio endpoint (POST)

In [ ]:
r = client.post('/api/v1/portfolio', json={'symbols': ['AAPL','MSFT','NVDA'], 'method': 'mean_variance'})
print(r.status_code, r.json())

## 5. Backtest endpoint

In [ ]:
r = client.get('/api/v1/backtest/NVDA')
print(r.status_code, list(r.json().get('comparison', {}).keys()))

## 6. Launch the real services
```bash
# API on :8000
uvicorn src.api.fastapi_app:app --host 0.0.0.0 --port 8000

# Dashboard on :8501
streamlit run frontend/dashboard.py
```
Docker alternative: `docker compose -f docker/docker-compose.yml up --build`